In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Set your base path — all other paths build from this
BASE = "/content/drive/MyDrive/FiQA"
print(f"Drive mounted. Base path: {BASE}")

Drive mounted. Base path: /content/drive/MyDrive/FiQA


In [3]:
#INSTALL LIBRARIES
# ============================================================
# rank_bm25  → BM25 algorithm implementation
# pytrec_eval → standard IR evaluation (NDCG, MRR, Recall)

!pip install -q rank_bm25 pytrec_eval

  Preparing metadata (setup.py) ... done


In [4]:
import pandas as pd
import numpy as np
import json
import os
import re
from rank_bm25 import BM25Okapi
import pytrec_eval
from tqdm import tqdm   # progress bar

In [6]:
#LOAD FIQA CORPUS
corpus_df = pd.read_csv(
    f"{BASE}/corpus.csv",
    usecols=["_id", "text"]
)

# Drop any rows where text is missing
corpus_df = corpus_df.dropna(subset=["text"]).reset_index(drop=True)

# Convert _id to string — important for matching with qrels later
corpus_df["_id"] = corpus_df["_id"].astype(str)

print(f"Corpus loaded: {len(corpus_df)} passages")
print(f"Sample passage:")
print(f"  ID  : {corpus_df['_id'][0]}")
print(f"  Text: {corpus_df['text'][0][:150]}...")

Corpus loaded: 57600 passages
Sample passage:
  ID  : 3
  Text: I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they...


In [7]:
#TOKENIZE CORPUS
# ============================================================
# BM25 works on lists of words, not raw strings
# Tokenization = lowercase + remove punctuation + split by space
#
# Before: "Cards are declined abroad when payments are disabled."
# After:  ["cards", "are", "declined", "abroad", "when", "payments", "are", "disabled"]
#
# tqdm shows a progress bar — 57,638 passages takes ~1 minute

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)  # remove punctuation
    return text.split()                         # split into word list

print("Tokenizing corpus... (this takes ~1-2 minutes)")
tokenized_corpus = [tokenize(text) for text in tqdm(corpus_df["text"])]
print(f"Tokenization complete: {len(tokenized_corpus)} passages tokenized")
print(f"Sample tokens: {tokenized_corpus[0][:10]}")

Tokenizing corpus... (this takes ~1-2 minutes)


100%|██████████| 57600/57600 [00:02<00:00, 20544.18it/s]

Tokenization complete: 57600 passages tokenized
Sample tokens: ['i', 'm', 'not', 'saying', 'i', 'don', 't', 'like', 'the', 'idea']


In [8]:
#BUILD BM25 INDEX
# ============================================================
# BM25Okapi builds the full inverted index over all passages
# Internally it calculates:
#   - Term frequency (TF) for each word in each passage
#   - Inverse document frequency (IDF) — rare words score higher
#   - Average document length — for length normalization
#
# This is the "index" that makes keyword search fast
# Takes ~2-3 minutes to build


bm25 = BM25Okapi(tokenized_corpus)

In [10]:
#LOAD TEST QUERIES AND QRELS
# ============================================================
# queries.csv has 6,648 total questions
# qrels_test.csv has relevance judgments for only 1,706 of them
# We only evaluate on queries that have ground truth (qrels)

print("Loading queries and qrels...")

# Load all queries
queries_df = pd.read_csv(f"{BASE}/queries.csv", usecols=["_id", "text"])
queries_df["_id"] = queries_df["_id"].astype(str)
print(f"Total queries loaded: {len(queries_df)}")

# Load test qrels — ground truth relevance judgments
# Columns: query-id, corpus-id, score
qrels_df = pd.read_csv(f"{BASE}/qrels_test.csv")
qrels_df.columns = [c.strip() for c in qrels_df.columns]  # strip whitespace
qrels_df = qrels_df.astype(str)
print(f"Qrels loaded: {len(qrels_df)} relevance pairs")
print(f"Qrels columns: {qrels_df.columns.tolist()}")

# Find which column names are used — FiQA uses query-id and corpus-id
query_col  = [c for c in qrels_df.columns if "query" in c.lower()][0]
corpus_col = [c for c in qrels_df.columns if "corpus" in c.lower()][0]
score_col  = [c for c in qrels_df.columns if "score" in c.lower()][0]
print(f"Detected columns → query: '{query_col}' | corpus: '{corpus_col}' | score: '{score_col}'")

# Get only the 1,706 queries that have test qrels
test_query_ids = qrels_df[query_col].unique().tolist()
test_queries_df = queries_df[queries_df["_id"].isin(test_query_ids)].reset_index(drop=True)
print(f"Test queries with qrels: {len(test_queries_df)}")

# Build qrels dict for pytrec_eval
# Format: {query_id: {doc_id: relevance_score}}
qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row[query_col])
    did = str(row[corpus_col])
    score = int(float(row[score_col]))
    if qid not in qrels_dict:
        qrels_dict[qid] = {}
    qrels_dict[qid][did] = score

print(f"Qrels dict built: {len(qrels_dict)} queries")

# Build corpus ID → index mapping for fast BM25 lookup
corpus_id_to_idx = {str(row["_id"]): idx for idx, row in corpus_df.iterrows()}

Loading queries and qrels...
Total queries loaded: 6648
Qrels loaded: 1706 relevance pairs
Qrels columns: ['query-id', 'corpus-id', 'score']
Detected columns → query: 'query-id' | corpus: 'corpus-id' | score: 'score'
Test queries with qrels: 648
Qrels dict built: 648 queries


In [11]:
#RUN BM25 RETRIEVAL
# ============================================================
# For each of the 1,706 test queries:
#   1. Tokenize the query
#   2. BM25 scores all 57,638 passages
#   3. Take top-10 highest scoring passage IDs
#   4. Store in results dict for evaluation
#
# Takes ~5-8 minutes for all 1,706 queries

TOP_K = 10

print(f"Running BM25 retrieval for {len(test_queries_df)} queries (top-{TOP_K})...")

results_dict = {}

for _, row in tqdm(test_queries_df.iterrows(), total=len(test_queries_df)):
    qid   = str(row["_id"])
    query = str(row["text"])

    # Step 1 — tokenize query same way as corpus
    tokenized_query = tokenize(query)

    # Step 2 — BM25 scores all passages
    scores = bm25.get_scores(tokenized_query)

    # Step 3 — get top-K passage indices
    top_k_indices = np.argsort(scores)[::-1][:TOP_K]

    # Step 4 — map indices back to passage IDs + scores
    results_dict[qid] = {
        str(corpus_df.iloc[idx]["_id"]): float(scores[idx])
        for idx in top_k_indices
    }

print(f"Retrieval complete: {len(results_dict)} queries processed")
print(f"\nSample result (first query):")
first_qid = list(results_dict.keys())[0]
print(f"  Query ID : {first_qid}")
print(f"  Query    : {test_queries_df[test_queries_df['_id']==first_qid]['text'].values[0][:100]}")
print(f"  Top-3 passages retrieved:")
for doc_id, score in list(results_dict[first_qid].items())[:3]:
    print(f"    doc_id={doc_id} | BM25 score={score:.4f}")

Running BM25 retrieval for 648 queries (top-10)...
This takes ~5-8 minutes...


100%|██████████| 648/648 [03:35<00:00,  3.00it/s]

Retrieval complete: 648 queries processed

Sample result (first query):
  Query ID : 4641
  Query    : Where should I park my rainy-day / emergency fund?
  Top-3 passages retrieved:
    doc_id=376148 | BM25 score=44.6926
    doc_id=580025 | BM25 score=27.5542
    doc_id=497993 | BM25 score=25.5488


In [12]:
#EVALUATE — NDCG@10, MRR, RECALL@10
# ============================================================
# pytrec_eval compares your retrieved results vs qrels ground truth
# and calculates standard IR metrics
#
# NDCG@10  — are correct passages near the top of your top-10?
# MRR      — where does the FIRST correct passage appear?
# Recall@10 — what fraction of correct passages did you retrieve?

# Define which metrics to calculate
evaluator = pytrec_eval.RelevanceEvaluator(
    qrels_dict,
    {"ndcg_cut_10", "recip_rank", "recall_10"}
)

# Run evaluation
eval_results = evaluator.evaluate(results_dict)

# Aggregate scores across all queries
ndcg_scores    = [v["ndcg_cut_10"] for v in eval_results.values()]
mrr_scores     = [v["recip_rank"]  for v in eval_results.values()]
recall_scores  = [v["recall_10"]   for v in eval_results.values()]

metrics = {
    "model":      "BM25 Baseline",
    "dataset":    "FiQA Test",
    "num_queries": len(eval_results),
    "NDCG@10":    round(np.mean(ndcg_scores),   4),
    "MRR":        round(np.mean(mrr_scores),    4),
    "Recall@10":  round(np.mean(recall_scores), 4),
}

print("\n" + "="*45)
print("      BM25 BASELINE RESULTS")
print("="*45)
print(f"  Queries evaluated : {metrics['num_queries']}")
print(f"  NDCG@10           : {metrics['NDCG@10']}")
print(f"  MRR               : {metrics['MRR']}")
print(f"  Recall@10         : {metrics['Recall@10']}")
print("="*45)


      BM25 BASELINE RESULTS
  Queries evaluated : 648
  NDCG@10           : 0.2169
  MRR               : 0.2706
  Recall@10         : 0.2784


In [13]:
#SAVE RESULTS TO DRIVE
# ============================================================
# Save 3 things:
# 1. baseline_metrics.json   → your thesis table numbers
# 2. baseline_results.csv    → top-10 passages per query
# 3. baseline_per_query.csv  → per-query breakdown for error analysis

results_dir = f"{BASE}/results"
os.makedirs(results_dir, exist_ok=True)

# Save 1 — metrics
metrics_path = f"{results_dir}/baseline_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved: {metrics_path}")

# Save 2 — top-10 results per query
rows = []
for qid, docs in results_dict.items():
    for rank, (doc_id, score) in enumerate(
        sorted(docs.items(), key=lambda x: x[1], reverse=True), 1
    ):
        rows.append({
            "query_id": qid,
            "rank":     rank,
            "doc_id":   doc_id,
            "score":    round(score, 4)
        })

results_df = pd.DataFrame(rows)
results_path = f"{results_dir}/baseline_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved: {results_path} ({len(results_df)} rows)")

# Save 3 — per query metrics
per_query_rows = []
for qid, scores in eval_results.items():
    query_text = test_queries_df[test_queries_df["_id"]==qid]["text"].values
    per_query_rows.append({
        "query_id":  qid,
        "query":     query_text[0] if len(query_text) > 0 else "",
        "NDCG@10":   round(scores["ndcg_cut_10"], 4),
        "MRR":       round(scores["recip_rank"],  4),
        "Recall@10": round(scores["recall_10"],   4),
    })

per_query_df = pd.DataFrame(per_query_rows).sort_values("NDCG@10", ascending=False)
per_query_path = f"{results_dir}/baseline_per_query.csv"
per_query_df.to_csv(per_query_path, index=False)
print(f"Saved: {per_query_path} ({len(per_query_df)} rows)")

print("\nAll results saved to Drive!")
print(f"\nYour baseline NDCG@10 = {metrics['NDCG@10']}")
print("This is the number everything else must beat.")

Saved: /content/drive/MyDrive/FiQA/results/baseline_metrics.json
Saved: /content/drive/MyDrive/FiQA/results/baseline_results.csv (6480 rows)
Saved: /content/drive/MyDrive/FiQA/results/baseline_per_query.csv (648 rows)

All results saved to Drive!

Your baseline NDCG@10 = 0.2169
This is the number everything else must beat.
